# Gene-set analyses of the comorbidity sub-graphs

The gene-set counterpart of `4_10`: instead of drawing the sub-graph around each interface protein, we turn it into a **gene set** and confront it with the bulk RNA-seq DE gene lists of `4_00`.

Two analyses, on two pairings.

**The analyses.** *GOAT* tests the sub-graph's gene set for enrichment against the whole effect-size ranking of a gene list (the R `goat` package, via `rpy2`). The *intersection* is its naive counterpart — how much of the gene set is simply among the genes the list marks significant. They want different namespaces: GOAT matches the gene lists' `gene` column, which is the HGNC dataset's `entrez_id`, so its gene sets are `ncbigene`; the intersection matches the `symbol` column, so its gene sets are `hgnc.symbol`.

**The pairings.** COVID → PD, whose downstream side is the PD activity-flow maps, and COVID → AD, whose downstream side is the **AD BEL knowledge graph** — the same two `4_10` draws, with the same collection names, walk and `max_levels`. The AD side used to be out of reach here: BEL nodes carried only `uniprot` annotations, and they were labelled `BELModelElement` but not `ModelElement`, which is what the annotation helpers in `commute_dm.queries` match — so they returned nothing for a BEL node, silently. `2_00` now writes `hgnc.symbol`, `ncbigene`, `hgnc` and `uniprot` on the KG's HGNC-encoded proteins and labels its nodes `ModelElement`, which is all it took.

Nothing here writes to the database, and nothing here loads a map: gene sets need no drawing material, so this is a matter of seconds rather than the couple of minutes `4_10` spends hydrating.

In [1]:
%store -r

Unable to restore variable 'PD_DM_AF_BUILD_DIR', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'CBM_KG_CYPHER_DATA_FILE', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'COVID_DM_AF_BUILD_DIR', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'INTERFACE_INTERSECTION_UPSTREAM_AND_DOWNSTREAM_ANALYSIS_DIR', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'INTERFACE_GOAT_UPSTREAM_ANALYSIS_DIR', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'AD_KG_DATA_DIR', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'COVID_KG_CYPHER_DATA_FILE', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'AD_KG_CYPHER_DATA_DIR', ignoring (use %store -d to forget!)
The error

In [2]:
import commute_dm.core
import commute_dm.submaps
import commute_dm.utils
import credentials
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session

In [3]:
backend = momapy_kb.lpg.backends.neo4j.Neo4jBackend(
    hostname=credentials.NEO4J_URI,
    username=credentials.NEO4J_USERNAME,
    password=credentials.NEO4J_PASSWORD,
    notifications_min_severity="off",
)
session = momapy_kb.lpg.session.Session(backend)

## Parameters

One entry per pairing, holding exactly what `4_10` holds in its own parameter cell — the collection names, the interface to join over and the levels to walk — plus the output directory of each analysis and mode.

`max_levels` differs between the two for the reason `4_10` gives: the AD influence graph is far denser than an activity-flow map, so a level means much more there. `WITH_SUBUNITS` now means the same thing on both sides — a CellDesigner complex's subunits, and a BEL complex's or composite's members. A BEL **activity** is not a subunit and is always followed: `act(p(X))` is X in another form, the way an active CellDesigner species is still that species.

In [4]:
MIN_N_NODES = 5
WITH_SUBUNITS = True
P_VALUE_CUTOFF = 0.05
MODES = ["upstream", "downstream", "upstream_and_downstream"]

PAIRINGS = [
    {
        "name": "COVID -> PD",
        # The interface is the three-way one, as in `4_10`: a protein has to be
        # shared with the AD KG as well to be looked at here.
        "upstream_collection_name": "COVID_DM_CD_AF",
        "downstream_collection_name": "PD_DM_CD_AF",
        "interface_collection_names": (
            "COVID_DM_CD_AF",
            "PD_DM_CD_AF",
            "AD_KG_CD_AF",
        ),
        "max_levels": [2, 3, 4, 5, 6],
        "goat_dir_paths": {
            "upstream": INTERFACE_PD_GOAT_UPSTREAM_ANALYSIS_DIR,
            "downstream": INTERFACE_PD_GOAT_DOWNSTREAM_ANALYSIS_DIR,
            "upstream_and_downstream": (
                INTERFACE_PD_GOAT_UPSTREAM_AND_DOWNSTREAM_ANALYSIS_DIR
            ),
        },
        "intersection_dir_paths": {
            "upstream": INTERFACE_PD_INTERSECTION_UPSTREAM_ANALYSIS_DIR,
            "downstream": INTERFACE_PD_INTERSECTION_DOWNSTREAM_ANALYSIS_DIR,
            "upstream_and_downstream": (
                INTERFACE_PD_INTERSECTION_UPSTREAM_AND_DOWNSTREAM_ANALYSIS_DIR
            ),
        },
    },
    {
        # Two-way, as in `4_10`: the pairing *is* COVID and the AD KG, which
        # `2_10` stores as an ordinary CellDesigner collection.
        "name": "COVID -> AD",
        "upstream_collection_name": "COVID_DM_CD_AF",
        "downstream_collection_name": "AD_KG_CD_AF",
        "interface_collection_names": (
            "COVID_DM_CD_AF",
            "AD_KG_CD_AF",
        ),
        "max_levels": [1, 2, 3],
        "goat_dir_paths": {
            "upstream": INTERFACE_AD_GOAT_UPSTREAM_ANALYSIS_DIR,
            "downstream": INTERFACE_AD_GOAT_DOWNSTREAM_ANALYSIS_DIR,
            "upstream_and_downstream": (
                INTERFACE_AD_GOAT_UPSTREAM_AND_DOWNSTREAM_ANALYSIS_DIR
            ),
        },
        "intersection_dir_paths": {
            "upstream": INTERFACE_AD_INTERSECTION_UPSTREAM_ANALYSIS_DIR,
            "downstream": INTERFACE_AD_INTERSECTION_DOWNSTREAM_ANALYSIS_DIR,
            "upstream_and_downstream": (
                INTERFACE_AD_INTERSECTION_UPSTREAM_AND_DOWNSTREAM_ANALYSIS_DIR
            ),
        },
    },
]

## The interface and the influence structure

For each pairing we compute its interface and load the influence structure the walk needs. `load_signed_influences` is a handful of small queries and hydrates no map: these analyses walk node ids and read annotations off the database, so the `source_map` that `4_10` pays a couple of minutes for would buy them nothing.

**Two different subunit options, and they do not fight.** `WITH_SUBUNITS` above is `gea`'s: it widens a selection to its species' subunits, and is what makes a subunit contribute its gene. The interface below is taken with subunits, which decides something else entirely -- which nodes a protein **seeds** its walks from. A protein stands on every species of its collection that is it, or that contains it, so a protein held only inside a complex stands on that complex and can seed a walk after all.


In [5]:
for pairing in PAIRINGS:
    pairing["interface"] = commute_dm.core.get_interface(
        session, pairing["interface_collection_names"]
    )
    pairing["influences"] = commute_dm.submaps.load_signed_influences(
        session,
        [
            pairing["upstream_collection_name"],
            pairing["downstream_collection_name"],
        ],
    )
    # Once per pairing, not once per mode: this is a query per interface protein.
    pairing["display_names"] = commute_dm.core.get_interface_display_names(
        session, pairing["interface"]
    )

{
    pairing["name"]: {
        "n_interface": len(pairing["interface"]),
    }
    for pairing in PAIRINGS
}

{'COVID -> PD': {'n_interface': 111}, 'COVID -> AD': {'n_interface': 159}}

## GOAT enrichment

For each pairing, each mode and each level, the gene set of the selection around every interface protein is tested against every gene list. Output is one CSV per gene list per level, plus a `summary.csv` listing, per interface protein, the levels at which it came out significant — ordered by how often it did.

The three modes are the three ways to read a comorbidity sub-graph: what the interface protein is downstream of in COVID (`upstream`), what it drives in the comorbid disease (`downstream`), and both at once.

`MIN_N_NODES` applies to **both** directions, so an interface protein is skipped at a level unless each side reaches at least that many nodes — which is why the counts above are much larger than the number of proteins actually tested.

In [6]:
for pairing in PAIRINGS:
    for mode in MODES:
        output_dir_path = pairing["goat_dir_paths"][mode]
        commute_dm.utils.remake_dir(output_dir_path)
        print(f"GOAT: {pairing['name']}, {mode} -> {output_dir_path}")
        commute_dm.core.make_goat_analysis_from_interface(
            session,
            pairing["interface"],
            pairing["influences"],
            GOAT_GENE_LISTS_BUILD_DIR,
            output_dir_path,
            upstream_collection_name=pairing["upstream_collection_name"],
            downstream_collection_name=pairing["downstream_collection_name"],
            mode=mode,
            max_levels=pairing["max_levels"],
            with_subunits=WITH_SUBUNITS,
            p_value_cutoff=P_VALUE_CUTOFF,
            min_n_nodes=MIN_N_NODES,
            display_names=pairing["display_names"],
        )

GOAT: COVID -> PD, upstream -> ../../build/results/interface/analysis/pd/goat_upstream


GOAT: COVID -> PD, downstream -> ../../build/results/interface/analysis/pd/goat_downstream


GOAT: COVID -> PD, upstream_and_downstream -> ../../build/results/interface/analysis/pd/goat_upstream_and_downstream


GOAT: COVID -> AD, upstream -> ../../build/results/interface/analysis/ad/goat_upstream


GOAT: COVID -> AD, downstream -> ../../build/results/interface/analysis/ad/goat_downstream


GOAT: COVID -> AD, upstream_and_downstream -> ../../build/results/interface/analysis/ad/goat_upstream_and_downstream


## Intersection with the significant genes

The same selections, read the naive way: how much of each sub-graph's gene set is among the genes a list marks significant. Gene sets are `hgnc.symbol` here rather than `ncbigene`, because that is what the gene lists' `symbol` column holds.

This is not a test — it has no null model and no p-value, and a big sub-graph will overlap more by construction, which is exactly what GOAT above corrects for. It is here because it says *which* genes, where GOAT only says whether. The `summary.csv` orders interface proteins by the mean `|intersection| / |gene set|` over the levels.

In [7]:
for pairing in PAIRINGS:
    for mode in MODES:
        output_dir_path = pairing["intersection_dir_paths"][mode]
        commute_dm.utils.remake_dir(output_dir_path)
        print(f"intersection: {pairing['name']}, {mode} -> {output_dir_path}")
        commute_dm.core.make_intersection_analysis_from_interface(
            session,
            pairing["interface"],
            pairing["influences"],
            GOAT_GENE_LISTS_BUILD_DIR,
            output_dir_path,
            upstream_collection_name=pairing["upstream_collection_name"],
            downstream_collection_name=pairing["downstream_collection_name"],
            mode=mode,
            max_levels=pairing["max_levels"],
            with_subunits=WITH_SUBUNITS,
            min_n_nodes=MIN_N_NODES,
            display_names=pairing["display_names"],
        )

intersection: COVID -> PD, upstream -> ../../build/results/interface/analysis/pd/intersection_upstream


intersection: COVID -> PD, downstream -> ../../build/results/interface/analysis/pd/intersection_downstream


intersection: COVID -> PD, upstream_and_downstream -> ../../build/results/interface/analysis/pd/intersection_upstream_and_downstream


intersection: COVID -> AD, upstream -> ../../build/results/interface/analysis/ad/intersection_upstream


intersection: COVID -> AD, downstream -> ../../build/results/interface/analysis/ad/intersection_downstream


intersection: COVID -> AD, upstream_and_downstream -> ../../build/results/interface/analysis/ad/intersection_upstream_and_downstream


## What came out

The head of each `summary.csv`: the interface proteins whose sub-graph was most often significant (GOAT) or most heavily overlapping (intersection), per pairing and mode.

Every output table is keyed by UniProt accession (`identifier`) with the HGNC symbol beside it (`display_name`) — the same name `4_10` gives the protein's map directory. The accession stays because it is what joins the collections and what is guaranteed unique: `BBC3` is the symbol of two accessions in the current data, and `get_interface_display_names` disambiguates those as `BBC3_<accession>` rather than letting two proteins collapse into one row.

In [8]:
import os.path

import pandas

for pairing in PAIRINGS:
    for analysis, dir_paths in (
        ("goat", pairing["goat_dir_paths"]),
        ("intersection", pairing["intersection_dir_paths"]),
    ):
        for mode in MODES:
            summary_file_path = os.path.join(dir_paths[mode], "summary.csv")
            if not os.path.exists(summary_file_path):
                continue
            summary_df = pandas.read_csv(summary_file_path, index_col=0)
            print(
                f"\n=== {pairing['name']} / {analysis} / {mode} "
                f"({len(summary_df)} proteins)"
            )
            display(summary_df.head(10))


=== COVID -> PD / goat / upstream (76 proteins)


,identifier,display_name,dgeobj_MS_brain_GSE123496.csv,dgeobj_COVID19_pbmc_GSE152418.csv,dgeobj_COVID19_cl_GSE147507.csv,deseqobj_PD_brain_GSE216281.csv,deseqobj_AD_brain_GSE95587.csv,dgeobj_PD_brain_GSE136666.csv,deseqobj_COVID19_hipsc_GSE179923.csv,deseqobj_COVID19_dopaminergic_GSE174745.csv,...,deseqobj_COVID19_pbmc_GSE251849.csv,deseqobj_COVID19_cl_GSE147507.csv,dgeobj_PD_brain_GSE68719.csv,dgeobj_PD_brain_GSE216281.csv,deseqobj_PD_brain_GSE136666.csv,dgeobj_COVID19_hipsc_GSE179923.csv,deseqobj_PD_brain_GSE68719.csv,dgeobj_COVID19_dopaminergic_GSE174745.csv,dgeobj_AD_brain_GSE95587.csv,deseqobj_COVID19_pbmc_GSE152418.csv
0,P99999,CYCS,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5, 6]",...,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],[],"[2, 3, 4, 5, 6]",[],[],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]"
1,P00403,MT-CO2,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[3, 4, 5, 6]",...,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],[],"[2, 3, 4, 5, 6]",[],[],"[3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]"
2,Q96KJ9,COX4I2,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[3, 4, 5, 6]",...,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],[],"[2, 3, 4, 5, 6]",[],[],"[3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]"
3,P10606,COX5B,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[3, 4, 5, 6]",...,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],[],"[2, 3, 4, 5, 6]",[],[],"[3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]"
4,P13073,COX4I1,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[3, 4, 5, 6]",...,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],[],"[2, 3, 4, 5, 6]",[],[],"[3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]"
5,P10176,COX8A,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[3, 4, 5, 6]",...,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],[],"[2, 3, 4, 5, 6]",[],[],"[3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]"
6,P20674,COX5A,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[3, 4, 5, 6]",...,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],[],"[2, 3, 4, 5, 6]",[],[],"[3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]"
7,P19404,NDUFV2,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],[],...,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],[],"[2, 3]",[],[],[],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]"
8,P28331,NDUFS1,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],[],...,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],[],"[2, 3]",[],[],[],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]"
9,P05231,IL6,"[3, 4, 5, 6]",[],"[4, 5, 6]",[5],"[3, 4, 5, 6]",[3],"[5, 6]",[],...,"[3, 4, 5, 6]","[4, 5, 6]","[3, 4, 5, 6]",[],[3],"[5, 6]","[3, 4, 5, 6]",[],"[3, 4, 5, 6]",[]



=== COVID -> PD / goat / downstream (76 proteins)


,identifier,display_name,dgeobj_MS_brain_GSE123496.csv,dgeobj_COVID19_pbmc_GSE152418.csv,dgeobj_COVID19_cl_GSE147507.csv,deseqobj_PD_brain_GSE216281.csv,deseqobj_AD_brain_GSE95587.csv,dgeobj_PD_brain_GSE136666.csv,deseqobj_COVID19_hipsc_GSE179923.csv,deseqobj_COVID19_dopaminergic_GSE174745.csv,...,deseqobj_COVID19_pbmc_GSE251849.csv,deseqobj_COVID19_cl_GSE147507.csv,dgeobj_PD_brain_GSE68719.csv,dgeobj_PD_brain_GSE216281.csv,deseqobj_PD_brain_GSE136666.csv,dgeobj_COVID19_hipsc_GSE179923.csv,deseqobj_PD_brain_GSE68719.csv,dgeobj_COVID19_dopaminergic_GSE174745.csv,dgeobj_AD_brain_GSE95587.csv,deseqobj_COVID19_pbmc_GSE152418.csv
0,P00403,MT-CO2,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[2],"[3, 4, 5, 6]","[4, 5, 6]","[2, 3]",[2],...,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],[],"[4, 5, 6]","[2, 3]",[],[2],"[4, 5, 6]","[2, 3, 4, 5, 6]"
1,P19404,NDUFV2,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],[],...,"[2, 3, 4, 6]","[2, 3, 4, 5, 6]",[],[],"[2, 3, 4, 5, 6]",[],[],[],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]"
2,P28331,NDUFS1,"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],[],...,"[2, 3, 4, 6]","[2, 3, 4, 5, 6]",[],[],"[2, 3, 4, 5, 6]",[],[],[],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]"
3,Q96KJ9,COX4I2,"[4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5, 6]","[3, 4, 5, 6]",[],[2],...,"[4, 5, 6]","[2, 3, 4, 5, 6]",[],[],"[4, 5, 6]",[],[],[2],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]"
4,P10606,COX5B,"[4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5, 6]","[3, 4, 5, 6]",[],[2],...,"[4, 5, 6]","[2, 3, 4, 5, 6]",[],[],"[4, 5, 6]",[],[],[2],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]"
5,P13073,COX4I1,"[4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5, 6]","[3, 4, 5, 6]",[],[2],...,"[4, 5, 6]","[2, 3, 4, 5, 6]",[],[],"[4, 5, 6]",[],[],[2],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]"
6,P10176,COX8A,"[4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5, 6]","[3, 4, 5, 6]",[],[2],...,"[4, 5, 6]","[2, 3, 4, 5, 6]",[],[],"[4, 5, 6]",[],[],[2],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]"
7,P20674,COX5A,"[4, 5, 6]","[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]",[],"[2, 3, 4, 5, 6]","[3, 4, 5, 6]",[],[2],...,"[4, 5, 6]","[2, 3, 4, 5, 6]",[],[],"[4, 5, 6]",[],[],[2],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]"
8,P99999,CYCS,"[4, 5]","[4, 5, 6]","[4, 5, 6]",[],"[2, 3, 4, 5, 6]","[4, 5, 6]",[],[2],...,[4],"[4, 5, 6]",[],[],"[4, 5, 6]",[],[],[2],"[2, 3, 4, 5, 6]","[4, 5, 6]"
9,O60603,TLR2,"[2, 3, 4, 5, 6]",[2],[],[6],"[2, 3, 4, 5, 6]",[],[],[],...,"[2, 3, 4, 6]",[],"[4, 5, 6]",[],[],[],"[4, 5, 6]",[],"[2, 3, 4, 5, 6]",[2]



=== COVID -> PD / goat / upstream_and_downstream (76 proteins)


...

,identifier,score,display_name,dgeobj_MS_brain_GSE123496.csv,dgeobj_COVID19_pbmc_GSE152418.csv,dgeobj_COVID19_cl_GSE147507.csv,deseqobj_PD_brain_GSE216281.csv,deseqobj_AD_brain_GSE95587.csv,dgeobj_PD_brain_GSE136666.csv,deseqobj_COVID19_hipsc_GSE179923.csv,...,deseqobj_COVID19_pbmc_GSE251849.csv,deseqobj_COVID19_cl_GSE147507.csv,dgeobj_PD_brain_GSE68719.csv,dgeobj_PD_brain_GSE216281.csv,deseqobj_PD_brain_GSE136666.csv,dgeobj_COVID19_hipsc_GSE179923.csv,deseqobj_PD_brain_GSE68719.csv,dgeobj_COVID19_dopaminergic_GSE174745.csv,dgeobj_AD_brain_GSE95587.csv,deseqobj_COVID19_pbmc_GSE152418.csv
0,Q14116,0.336667,IL18,"['2: 4/5', '3: 4/6']","['2: 2/5', '3: 2/6']","['2: 1/5', '3: 2/6']","['2: 0/5', '3: 0/6']","['2: 4/5', '3: 5/6']","['2: 0/5', '3: 0/6']","['2: 0/5', '3: 0/6']",...,"['2: 1/5', '3: 1/6']","['2: 1/5', '3: 2/6']","['2: 5/5', '3: 6/6']","['2: 0/5', '3: 0/6']","['2: 1/5', '3: 1/6']","['2: 0/5', '3: 0/6']","['2: 4/5', '3: 5/6']","['2: 1/5', '3: 1/6']","['2: 3/5', '3: 4/6']","['2: 2/5', '3: 2/6']"
1,P01137,0.310000,TGFB1,"['1: 0/5', '2: 0/5', '3: 0/5']","['1: 3/5', '2: 3/5', '3: 3/5']","['1: 1/5', '2: 1/5', '3: 1/5']","['1: 0/5', '2: 0/5', '3: 0/5']","['1: 5/5', '2: 5/5', '3: 5/5']","['1: 0/5', '2: 0/5', '3: 0/5']","['1: 0/5', '2: 0/5', '3: 0/5']",...,"['1: 1/5', '2: 1/5', '3: 1/5']","['1: 1/5', '2: 1/5', '3: 1/5']","['1: 4/5', '2: 4/5', '3: 4/5']","['1: 0/5', '2: 0/5', '3: 0/5']","['1: 0/5', '2: 0/5', '3: 0/5']","['1: 0/5', '2: 0/5', '3: 0/5']","['1: 4/5', '2: 4/5', '3: 4/5']","['1: 2/5', '2: 2/5', '3: 2/5']","['1: 5/5', '2: 5/5', '3: 5/5']","['1: 3/5', '2: 3/5', '3: 3/5']"
2,Q00535,0.300000,CDK5,['3: 0/1'],['3: 1/1'],['3: 1/1'],['3: 0/1'],['3: 1/1'],['3: 0/1'],['3: 0/1'],...,['3: 0/1'],['3: 1/1'],['3: 0/1'],['3: 0/1'],['3: 0/1'],['3: 0/1'],['3: 0/1'],['3: 0/1'],['3: 1/1'],['3: 1/1']
3,Q07817,0.285000,BCL2L1,"['1: 2/4', '2: 3/5', '3: 3/5']","['1: 2/4', '2: 2/5', '3: 2/5']","['1: 0/4', '2: 0/5', '3: 0/5']","['1: 0/4', '2: 0/5', '3: 0/5']","['1: 3/4', '2: 4/5', '3: 4/5']","['1: 0/4', '2: 0/5', '3: 0/5']","['1: 0/4', '2: 0/5', '3: 0/5']",...,"['1: 1/4', '2: 1/5', '3: 1/5']","['1: 0/4', '2: 0/5', '3: 0/5']","['1: 3/4', '2: 4/5', '3: 4/5']","['1: 0/4', '2: 0/5', '3: 0/5']","['1: 0/4', '2: 0/5', '3: 0/5']","['1: 0/4', '2: 0/5', '3: 0/5']","['1: 3/4', '2: 4/5', '3: 4/5']","['1: 0/4', '2: 0/5', '3: 0/5']","['1: 3/4', '2: 4/5', '3: 4/5']","['1: 2/4', '2: 2/5', '3: 2/5']"
4,P35228,0.273125,NOS2,"['2: 3/5', '3: 4/8']","['2: 2/5', '3: 2/8']","['2: 1/5', '3: 1/8']","['2: 0/5', '3: 0/8']","['2: 3/5', '3: 4/8']","['2: 0/5', '3: 0/8']","['2: 0/5', '3: 0/8']",...,"['2: 1/5', '3: 1/8']","['2: 1/5', '3: 1/8']","['2: 3/5', '3: 5/8']","['2: 0/5', '3: 0/8']","['2: 1/5', '3: 1/8']","['2: 0/5', '3: 0/8']","['2: 3/5', '3: 5/8']","['2: 1/5', '3: 2/8']","['2: 3/5', '3: 5/8']","['2: 3/5', '3: 4/8']"
5,P10599,0.272500,TXN,"['2: 0/2', '3: 1/5']","['2: 1/2', '3: 2/5']","['2: 0/2', '3: 2/5']","['2: 0/2', '3: 0/5']","['2: 2/2', '3: 3/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 0/5']",...,"['2: 0/2', '3: 0/5']","['2: 1/2', '3: 3/5']","['2: 0/2', '3: 2/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 2/5']","['2: 2/2', '3: 2/5']","['2: 2/2', '3: 3/5']","['2: 1/2', '3: 3/5']"
6,Q99683,0.271429,MAP3K5,"['2: 2/7', '3: 2/7']","['2: 4/7', '3: 4/7']","['2: 2/7', '3: 2/7']","['2: 0/7', '3: 0/7']","['2: 4/7', '3: 4/7']","['2: 0/7', '3: 0/7']","['2: 0/7', '3: 0/7']",...,"['2: 0/7', '3: 0/7']","['2: 2/7', '3: 2/7']","['2: 5/7', '3: 5/7']","['2: 0/7', '3: 0/7']","['2: 0/7', '3: 0/7']","['2: 0/7', '3: 0/7']","['2: 5/7', '3: 5/7']","['2: 1/7', '3: 1/7']","['2: 4/7', '3: 4/7']","['2: 5/7', '3: 5/7']"
7,P10415,0.269062,BCL2,"['2: 3/10', '3: 3/16']","['2: 2/10', '3: 6/16']","['2: 2/10', '3: 3/16']","['2: 0/10', '3: 0/16']","['2: 8/10', '3: 11/16']","['2: 0/10', '3: 0/16']","['2: 0/10', '3: 0/16']",...,"['2: 1/10', '3: 1/16']","['2: 2/10', '3: 3/16']","['2: 9/10', '3: 10/16']","['2: 0/10', '3: 0/16']","['2: 0/10', '3: 0/16']","['2: 0/10


=== COVID -> AD / intersection / downstream (77 proteins)


,identifier,score,display_name,dgeobj_MS_brain_GSE123496.csv,dgeobj_COVID19_pbmc_GSE152418.csv,dgeobj_COVID19_cl_GSE147507.csv,deseqobj_PD_brain_GSE216281.csv,deseqobj_AD_brain_GSE95587.csv,dgeobj_PD_brain_GSE136666.csv,deseqobj_COVID19_hipsc_GSE179923.csv,...,deseqobj_COVID19_pbmc_GSE251849.csv,deseqobj_COVID19_cl_GSE147507.csv,dgeobj_PD_brain_GSE68719.csv,dgeobj_PD_brain_GSE216281.csv,deseqobj_PD_brain_GSE136666.csv,dgeobj_COVID19_hipsc_GSE179923.csv,deseqobj_PD_brain_GSE68719.csv,dgeobj_COVID19_dopaminergic_GSE174745.csv,dgeobj_AD_brain_GSE95587.csv,deseqobj_COVID19_pbmc_GSE152418.csv
0,Q07812,0.316667,BAX,"['2: 1/3', '3: 1/3']","['2: 2/3', '3: 2/3']","['2: 2/3', '3: 2/3']","['2: 0/3', '3: 0/3']","['2: 1/3', '3: 1/3']","['2: 0/3', '3: 0/3']","['2: 0/3', '3: 0/3']",...,"['2: 1/3', '3: 1/3']","['2: 2/3', '3: 2/3']","['2: 2/3', '3: 2/3']","['2: 0/3', '3: 0/3']","['2: 0/3', '3: 0/3']","['2: 0/3', '3: 0/3']","['2: 2/3', '3: 2/3']","['2: 1/3', '3: 1/3']","['2: 1/3', '3: 1/3']","['2: 2/3', '3: 2/3']"
1,P01100,0.267500,FOS,"['2: 2/2', '3: 2/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 0/5']","['2: 2/2', '3: 4/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 0/5']",...,"['2: 0/2', '3: 0/5']","['2: 1/2', '3: 1/5']","['2: 2/2', '3: 3/5']","['2: 0/2', '3: 0/5']","['2: 0/2', '3: 1/5']","['2: 0/2', '3: 0/5']","['2: 2/2', '3: 4/5']","['2: 0/2', '3: 0/5']","['2: 2/2', '3: 4/5']","['2: 0/2', '3: 0/5']"
2,P05362,0.258333,ICAM1,"['2: 1/1', '3: 5/6']","['2: 0/1', '3: 1/6']","['2: 1/1', '3: 1/6']","['2: 0/1', '3: 0/6']","['2: 0/1', '3: 1/6']","['2: 0/1', '3: 0/6']","['2: 0/1', '3: 0/6']",...,"['2: 0/1', '3: 0/6']","['2: 1/1', '3: 1/6']","['2: 1/1', '3: 3/6']","['2: 0/1', '3: 0/6']","['2: 0/1', '3: 1/6']","['2: 0/1', '3: 0/6']","['2: 1/1', '3: 3/6']","['2: 0/1', '3: 1/6']","['2: 0/1', '3: 1/6']","['2: 0/1', '3: 2/6']"
3,Q14116,0.250000,IL18,"['2: 1/1', '3: 1/2']","['2: 0/1', '3: 1/2']","['2: 0/1', '3: 0/2']","['2: 0/1', '3: 0/2']","['2: 1/1', '3: 1/2']","['2: 0/1', '3: 0/2']","['2: 0/1', '3: 0/2']",...,"['2: 0/1', '3: 0/2']","['2: 0/1', '3: 0/2']","['2: 1/1', '3: 1/2']","['2: 0/1', '3: 0/2']","['2: 0/1', '3: 0/2']","['2: 0/1', '3: 0/2']","['2: 1/1', '3: 1/2']","['2: 0/1', '3: 0/2']","['2: 1/1', '3: 1/2']","['2: 0/1', '3: 1/2']"
4,P04637,0.231667,TP53,"['2: 2/3', '3: 2/5']","['2: 1/3', '3: 2/5']","['2: 1/3', '3: 1/5']","['2: 0/3', '3: 0/5']","['2: 1/3', '3: 3/5']","['2: 0/3', '3: 0/5']","['2: 0/3', '3: 0/5']",...,"['2: 0/3', '3: 0/5']","['2: 1/3', '3: 1/5']","['2: 2/3', '3: 2/5']","['2: 0/3', '3: 0/5']","['2: 0/3', '3: 0/5']","['2: 0/3', '3: 0/5']","['2: 1/3', '3: 1/5']","['2: 1/3', '3: 2/5']","['2: 1/3', '3: 3/5']","['2: 1/3', '3: 3/5']"
5,P05412,0.218254,JUN,"['1: 1/3', '2: 1/3', '3: 22/105']","['1: 1/3', '2: 1/3', '3: 25/105']","['1: 2/3', '2: 2/3', '3: 15/105']","['1: 0/3', '2: 0/3', '3: 0/105']","['1: 2/3', '2: 2/3', '3: 35/105']","['1: 0/3', '2: 0/3', '3: 0/105']","['1: 0/3', '2: 0/3', '3: 0/105']",...,"['1: 0/3', '2: 0/3', '3: 1/105']","['1: 2/3', '2: 2/3', '3: 14/105']","['1: 0/3', '2: 0/3', '3: 37/105']","['1: 0/3', '2: 0/3', '3: 0/105']","['1: 0/3', '2: 0/3', '3: 5/105']","['1: 0/3', '2: 0/3', '3: 0/105']","['1: 1/3', '2: 1/3', '3: 41/105']","['1: 0/3', '2: 0/3', '3: 13/105']","['1: 2/3', '2: 2/3', '3: 37/105']","['1: 1/3', '2: 1/3', '3: 33/105']"
6,P40763,0.214316,STAT3,"['2: 4/9', '3: 8/26']","['2: 3/9', '3: 9/26']","['2: 3/9', '3: 6/26']","['2: 0/9', '3: 0/26']","['2: 5/9', '3: 8/26']","['2: 0/9', '3: 0/26']","['2: 0/9', '3: 0/26']",...,"['2: 2/9', '3: 4/26']","['2: 3/9', '3: 6/26']","['2: 6/9', '3: 8/26']","['2: 0/9', '3: 0/26']","['2: 0/9', '3: 0/26']","['2: 0/9', '3: 0/26']","['2: 6/9', '3: 8/26']","['2: 1/9', '3: 1/26']","['2: 5/9', '3: 8/26']","['2: 3/9', '3: 11/26']"
7,Q12933,0.208333,TRAF2,"['2: 1/2', '3: 1/3']","['2: 1/2', '3: 1/3']","['2: 0/2', '3: 0/3']","['2: 0/2', '3: 0/3']","['2: 1/2', '3: 1/3']","['2: 0/2', '3: 0/3']","['2: 0/2', '3: 0/3']",...,"['2: 0/2', '3: 0/3']","['2: 0/2', '3: 0/3']","['2: 1/2', '


=== COVID -> AD / intersection / upstream_and_downstream (77 proteins)


,identifier,score,display_name,dgeobj_MS_brain_GSE123496.csv,dgeobj_COVID19_pbmc_GSE152418.csv,dgeobj_COVID19_cl_GSE147507.csv,deseqobj_PD_brain_GSE216281.csv,deseqobj_AD_brain_GSE95587.csv,dgeobj_PD_brain_GSE136666.csv,deseqobj_COVID19_hipsc_GSE179923.csv,...,deseqobj_COVID19_pbmc_GSE251849.csv,deseqobj_COVID19_cl_GSE147507.csv,dgeobj_PD_brain_GSE68719.csv,dgeobj_PD_brain_GSE216281.csv,deseqobj_PD_brain_GSE136666.csv,dgeobj_COVID19_hipsc_GSE179923.csv,deseqobj_PD_brain_GSE68719.csv,dgeobj_COVID19_dopaminergic_GSE174745.csv,dgeobj_AD_brain_GSE95587.csv,deseqobj_COVID19_pbmc_GSE152418.csv
0,Q14116,0.320000,IL18,"['2: 4/5', '3: 4/7']","['2: 2/5', '3: 3/7']","['2: 1/5', '3: 2/7']","['2: 0/5', '3: 0/7']","['2: 4/5', '3: 5/7']","['2: 0/5', '3: 0/7']","['2: 0/5', '3: 0/7']",...,"['2: 1/5', '3: 1/7']","['2: 1/5', '3: 2/7']","['2: 5/5', '3: 6/7']","['2: 0/5', '3: 0/7']","['2: 1/5', '3: 1/7']","['2: 0/5', '3: 0/7']","['2: 4/5', '3: 5/7']","['2: 1/5', '3: 1/7']","['2: 3/5', '3: 4/7']","['2: 2/5', '3: 3/7']"
1,P04637,0.248438,TP53,"['2: 2/8', '3: 2/16']","['2: 1/8', '3: 6/16']","['2: 3/8', '3: 4/16']","['2: 0/8', '3: 0/16']","['2: 5/8', '3: 10/16']","['2: 0/8', '3: 0/16']","['2: 0/8', '3: 0/16']",...,"['2: 0/8', '3: 0/16']","['2: 3/8', '3: 4/16']","['2: 7/8', '3: 8/16']","['2: 0/8', '3: 0/16']","['2: 0/8', '3: 0/16']","['2: 0/8', '3: 0/16']","['2: 6/8', '3: 7/16']","['2: 3/8', '3: 6/16']","['2: 5/8', '3: 11/16']","['2: 1/8', '3: 7/16']"
2,P01100,0.233611,FOS,"['2: 5/15', '3: 10/36']","['2: 3/15', '3: 13/36']","['2: 4/15', '3: 11/36']","['2: 0/15', '3: 0/36']","['2: 7/15', '3: 16/36']","['2: 0/15', '3: 0/36']","['2: 0/15', '3: 0/36']",...,"['2: 0/15', '3: 1/36']","['2: 5/15', '3: 12/36']","['2: 9/15', '3: 19/36']","['2: 0/15', '3: 0/36']","['2: 0/15', '3: 1/36']","['2: 0/15', '3: 0/36']","['2: 10/15', '3: 21/36']","['2: 4/15', '3: 7/36']","['2: 8/15', '3: 18/36']","['2: 4/15', '3: 15/36']"
3,Q12933,0.231944,TRAF2,"['2: 3/12', '3: 3/18']","['2: 5/12', '3: 7/18']","['2: 6/12', '3: 8/18']","['2: 0/12', '3: 0/18']","['2: 6/12', '3: 8/18']","['2: 0/12', '3: 0/18']","['2: 0/12', '3: 0/18']",...,"['2: 0/12', '3: 0/18']","['2: 6/12', '3: 8/18']","['2: 6/12', '3: 8/18']","['2: 0/12', '3: 0/18']","['2: 0/12', '3: 0/18']","['2: 0/12', '3: 0/18']","['2: 7/12', '3: 9/18']","['2: 1/12', '3: 3/18']","['2: 7/12', '3: 8/18']","['2: 6/12', '3: 9/18']"
4,P25963,0.227691,NFKBIA,"['2: 5/18', '3: 10/32']","['2: 7/18', '3: 12/32']","['2: 4/18', '3: 6/32']","['2: 0/18', '3: 0/32']","['2: 10/18', '3: 12/32']","['2: 0/18', '3: 0/32']","['2: 0/18', '3: 0/32']",...,"['2: 1/18', '3: 2/32']","['2: 3/18', '3: 5/32']","['2: 10/18', '3: 17/32']","['2: 0/18', '3: 0/32']","['2: 1/18', '3: 1/32']","['2: 0/18', '3: 0/32']","['2: 10/18', '3: 19/32']","['2: 6/18', '3: 6/32']","['2: 11/18', '3: 14/32']","['2: 7/18', '3: 13/32']"
5,Q07812,0.224123,BAX,"['2: 4/15', '3: 6/19']","['2: 6/15', '3: 8/19']","['2: 4/15', '3: 5/19']","['2: 0/15', '3: 0/19']","['2: 5/15', '3: 8/19']","['2: 0/15', '3: 0/19']","['2: 0/15', '3: 0/19']",...,"['2: 2/15', '3: 2/19']","['2: 4/15', '3: 5/19']","['2: 9/15', '3: 11/19']","['2: 0/15', '3: 0/19']","['2: 0/15', '3: 0/19']","['2: 0/15', '3: 0/19']","['2: 9/15', '3: 11/19']","['2: 2/15', '3: 3/19']","['2: 6/15', '3: 9/19']","['2: 6/15', '3: 8/19']"
6,P10415,0.222147,BCL2,"['2: 3/16', '3: 3/23']","['2: 2/16', '3: 6/23']","['2: 2/16', '3: 3/23']","['2: 0/16', '3: 0/23']","['2: 11/16', '3: 14/23']","['2: 0/16', '3: 0/23']","['2: 0/16', '3: 0/23']",...,"['2: 1/16', '3: 1/23']","['2: 2/16', '3: 3/23']","['2: 13/16', '3: 14/23']","['2: 0/16', '3: 0/23']","['2: 0/16', '3: 0/23']","['2: 0/16', '3: 0/23']","['2: 13/16', '3: 14/23']","['2: 4/16', '3: 6/23']","['2: 10/16', '3: 14/23']","['2: 2/16', '3: 6/23']"
7,P99999,0.209001,CYCS,"['1: 1/18', '2: 14/140', '3: 30/219']","['1: 8/18', '2: 83/140', '3: 102/219']","['1: 4/18', '2: 32/140', '3: 39/219']","['1: 0/18', '2: 0/140', '3: 0/219']","['1: 10/18', '2: 73/140', '3: 94/219']","['1: 0/18', '2: 0/140', '3: 0/219'